## Obteniendo el ranking FIFA

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from io import StringIO
import time

print("Iniciando motor de Chrome")

servicio = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=servicio)

url_fifa = "https://inside.fifa.com/es/fifa-world-ranking/men"
driver.get(url_fifa)

time.sleep(6) 

try:
    print("Buscando el botón para desplegar todo el ranking...")
    
    # Buscamos un botón cuyo texto contenga "completa" o "Mostrar" o "Show full"
    xpath_boton = "//*[contains(text(), 'completa') or contains(text(), 'Mostrar') or contains(text(), 'Show full')]"
    boton = driver.find_element(By.XPATH, xpath_boton)
    
    driver.execute_script("arguments[0].click();", boton)
    
    print("Botón presionado.")
    time.sleep(5) # Le damos tiempo a que aparezca el resto de la tabla
except Exception as e:
    print("No encontramos el botón o ya estaba todo cargado. Extraeremos lo que esté visible.")

# ------------------------------------------

# Tomamos la "foto" final con los datos desplegados
html_cargado = driver.page_source
html_seguro = StringIO(html_cargado)
tablas_ranking = pd.read_html(html_seguro)

driver.quit()
print("Navegador cerrado.\n")

if len(tablas_ranking) > 0:
    df_ranking_fifa = tablas_ranking[0]
    print(f"ÉXITO: Se capturó una tabla con {len(df_ranking_fifa)} países. 🏆")
    display(df_ranking_fifa.head(15))
else:
    print("No se logró extraer ninguna tabla.")

Iniciando motor de Chrome
Buscando el botón para desplegar todo el ranking...
Botón presionado.
Navegador cerrado.

ÉXITO: Se capturó una tabla con 211 países. 🏆


,Rango,Equipo,Partidos,Partido +/-,Puntos,Más
0,10,Argentina,ArgentinaAustriaFINAL20,12.87,1901.93,NaN
1,21,Francia,NoruegaFrancia26 jun13:00,NaN,1894.40,NaN
2,31,España,UruguayEspaña26 jun18:00,NaN,1864.32,NaN
3,40,Inglaterra,InglaterraGhanaFINAL00,-17.86,1829.82,NaN
4,51,Brasil,EscociaBrasilFINAL03,13.18,1785.19,NaN
5,61,Marruecos,MarruecosHaitíFINAL42,6.42,1776.40,NaN
6,72,Portugal,PortugalUzbekistánFINAL50,11.64,1766.74,NaN
7,80,Países Bajos,TúnezPaíses Bajos25 jun17:00,NaN,1764.40,NaN
8,91,Alemania,EcuadorAlemania25 jun14:00,NaN,1760.46,NaN
9,104,México,ChequiaMéxicoFINAL03,14.23,1736.01,NaN


In [3]:
traduccion_paises_mundial_2026 = {
    # Grupo A
    'México': 'Mexico',
    'Sudáfrica': 'South Africa',
    'Corea del Sur': 'South Korea',
    'Chequia': 'Czech Republic',
    'República Checa': 'Czech Republic',
    
    # Grupo B
    'Canadá': 'Canada',
    'Suiza': 'Switzerland',
    'Bosnia y Herzegovina': 'Bosnia and Herzegovina',
    'Catar': 'Qatar',
    
    # Grupo C
    'Brasil': 'Brazil',
    'Marruecos': 'Morocco',
    'Escocia': 'Scotland',
    'Haití': 'Haiti',
    
    # Grupo D
    'Estados Unidos': 'United States',
    'Australia': 'Australia',
    'Paraguay': 'Paraguay',
    'Turquía': 'Turkey', 
    
    # Grupo E
    'Alemania': 'Germany',
    'Costa de Marfil': 'Ivory Coast',
    'Ecuador': 'Ecuador',
    'Curazao': 'Curacao',
    
    # Grupo F
    'Países Bajos': 'Netherlands',
    'Holanda': 'Netherlands', # Por si acaso
    'Japón': 'Japan',
    'Suecia': 'Sweden',
    'Túnez': 'Tunisia',
    
    # Grupo G
    'Egipto': 'Egypt',
    'Irán': 'Iran',
    'Bélgica': 'Belgium',
    'Nueva Zelanda': 'New Zealand',
    
    # Grupo H
    'España': 'Spain',
    'Uruguay': 'Uruguay',
    'Cabo Verde': 'Cape Verde',
    'Arabia Saudita': 'Saudi Arabia',
    'Arabia Saudí': 'Saudi Arabia', # Variación común
    
    # Grupo I
    'Francia': 'France',
    'Noruega': 'Norway',
    'Senegal': 'Senegal',
    'Irak': 'Iraq',
    
    # Grupo J
    'Argentina': 'Argentina',
    'Austria': 'Austria',
    'Argelia': 'Algeria',
    'Jordania': 'Jordan',
    
    # Grupo K
    'Portugal': 'Portugal',
    'Colombia': 'Colombia',
    'RD Congo': 'DR Congo',
    'República Democrática del Congo': 'DR Congo',
    'Uzbekistán': 'Uzbekistan',
    
    # Grupo L
    'Inglaterra': 'England',
    'Ghana': 'Ghana',
    'Panamá': 'Panama',
    'Croacia': 'Croatia'
}


In [4]:
ponderaciones = df_ranking_fifa.copy()
ponderaciones = ponderaciones[['Equipo','Puntos']]
ponderaciones['Equipo'] = ponderaciones['Equipo'].replace(traduccion_paises_mundial_2026)
display(ponderaciones.head(),ponderaciones.tail())

,Equipo,Puntos
0,Argentina,1901.93
1,France,1894.40
2,Spain,1864.32
3,England,1829.82
4,Brazil,1785.19


,Equipo,Puntos
206,Bahamas,786.82
207,Islas Vírgenes Estadounidenses,779.76
208,Islas Vírgenes Británicas,777.41
209,Anguilla,760.25
210,San Marino,721.20


## Obteniendo estadísticas de los partidos hasta el momento

In [6]:
print("Iniciando nuestro Chrome automatizado...")

# 1. EL MOTOR: Instalamos y configuramos el driver para controlar Chrome
servicio = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=servicio)

# 2. LA NAVEGACIÓN: Le decimos al robot a dónde ir
url_espn = "https://espndeportes.espn.com/futbol/posiciones/_/liga/fifa.world"
driver.get(url_espn)

# 3. LA PACIENCIA: Le damos 5 segundos a ESPN para que su JavaScript cargue los datos
time.sleep(5)

print("Extrayendo el código HTML procesado por Selenium...")

# 1. Tomamos la "foto" del código fuente tal cual se ve en el navegador ahora mismo
html_cargado = driver.page_source

# 2. Usamos a nuestro viejo amigo Pandas para buscar las tablas
html_seguro = StringIO(html_cargado)
tablas_espn = pd.read_html(html_seguro)

# 3. Según tu captura, la primera tabla es la del Grupo A
if len(tablas_espn) > 0:
    print(f"Pandas encontró {len(tablas_espn)} tablas.")    
else:
    print("No se encontraron tablas. Revisa si la página cargó bien.")

# 4. LA REGLA DE ORO: Siempre cerramos el navegador al terminar de usarlo
driver.quit()
print("Navegador automatizado cerrado.")


Iniciando nuestro Chrome automatizado...
Extrayendo el código HTML procesado por Selenium...
Pandas encontró 2 tablas.
Navegador automatizado cerrado.


In [7]:
grupos=[]
resultados=[]
tablas_espn[0].columns=['Equipo']
tablas_espn[1].columns=tablas_espn[1].iloc[0]
for i in range(1,len(tablas_espn[0]),5):
    grupos.append(tablas_espn[0].iloc[i:i+4])
    resultados.append(tablas_espn[1].iloc[i:i+4])

grupos = pd.concat(grupos)
resultados = pd.concat(resultados).reset_index(drop=True)

In [8]:
grupos_limpios=grupos['Equipo'].str[4:].replace(traduccion_paises_mundial_2026).reset_index(drop=True)

In [9]:
df_completo = pd.concat([grupos_limpios,resultados],axis = 1)
display(df_completo.head(),df_completo.tail())

,Equipo,J,G,E,P,GF,GC,DIF,PTS
0,Mexico,3,3,0,0,6,0,+6,9
1,South Africa,3,1,1,1,2,3,-1,4
2,South Korea,3,1,0,2,2,3,-1,3
3,Czech Republic,3,0,1,2,2,6,-4,1
4,Switzerland,3,2,1,0,7,3,+4,7


,Equipo,J,G,E,P,GF,GC,DIF,PTS
43,Uzbekistan,2,0,0,2,1,8,-7,0
44,England,2,1,1,0,4,2,+2,4
45,Ghana,2,1,1,0,1,0,+1,4
46,Croatia,2,1,0,1,3,4,-1,3
47,Panama,2,0,0,2,0,2,-2,0


## Cálculo de métrica de forma actual

In [11]:
diccionario_excepciones = {'República de Corea':'South Korea'
                          ,'EE. UU.': 'United States'
                          ,'RI de Irán':'Iran'
                          ,'Islas de Cabo Verde':'Cape Verde'}
ponderaciones['Equipo']=ponderaciones['Equipo'].replace(diccionario_excepciones)

In [12]:
# 1. FILTRAR: Nos quedamos solo con los países que existen en tu tabla del mundial
equipos_mundiales = df_completo['Equipo'].unique()
ponderaciones_mundial = ponderaciones[ponderaciones['Equipo'].isin(equipos_mundiales)].copy()

print(f"Ranking filtrado: Tenemos {len(ponderaciones_mundial)} equipos listos.")

# 2. FUSIONAR: Pegamos los puntos FIFA al lado de las estadísticas del grupo
df_modelo = pd.merge(df_completo, ponderaciones_mundial, on='Equipo', how='inner')

Ranking filtrado: Tenemos 48 equipos listos.


In [13]:
display(df_modelo.head(),df_modelo.tail())

,Equipo,J,G,E,P,GF,GC,DIF,PTS,Puntos
0,Mexico,3,3,0,0,6,0,+6,9,1736.01
1,South Africa,3,1,1,1,2,3,-1,4,1451.24
2,South Korea,3,1,0,2,2,3,-1,3,1558.72
3,Czech Republic,3,0,1,2,2,6,-4,1,1467.26
4,Switzerland,3,2,1,0,7,3,+4,7,1676.00


,Equipo,J,G,E,P,GF,GC,DIF,PTS,Puntos
43,Uzbekistan,2,0,0,2,1,8,-7,0,1432.84
44,England,2,1,1,0,4,2,+2,4,1829.82
45,Ghana,2,1,1,0,1,0,+1,4,1398.57
46,Croatia,2,1,0,1,3,4,-1,3,1711.48
47,Panama,2,0,0,2,0,2,-2,0,1489.05


In [14]:
df_goles = df_modelo[['Equipo','GF','GC','J']].copy()
df_goles.to_csv("goles_dinamicos.csv",index=False, encoding='utf-8-sig')

In [15]:
df_modelo['Grupo_ID'] = df_modelo.index // 4

fuerza_por_grupo = df_modelo.groupby('Grupo_ID')['Puntos'].transform('sum')

puntos_rivales = fuerza_por_grupo - df_modelo['Puntos']
df_modelo['Promedio_Rivales'] = puntos_rivales / 3

# Puntos del torneo * (Fuerza promedio de los rivales / 1000)
df_modelo['Forma_Actual'] = round((df_modelo['PTS'].to_numpy(dtype=float)/df_modelo['J'].to_numpy(dtype=float)) * df_modelo['Promedio_Rivales'] / 1000,2)

columnas_finales = ['Equipo', 'PTS', 'Puntos', 'Promedio_Rivales', 'Forma_Actual']
df_final = df_modelo[columnas_finales].sort_values(by='Forma_Actual', ascending=False).reset_index(drop=True)

print("Métrica 'Forma Actual' calculada con éxito\n")
display(df_final.head(10))



Métrica 'Forma Actual' calculada con éxito



,Equipo,PTS,Puntos,Promedio_Rivales,Forma_Actual
0,Norway,6,1606.48,1650.666667,4.95
1,Colombia,6,1727.42,1557.316667,4.67
2,France,6,1894.40,1554.693333,4.66
3,United States,6,1709.59,1550.690000,4.65
4,Argentina,6,1901.93,1510.506667,4.53
5,Mexico,9,1736.01,1492.406667,4.48
6,Germany,6,1760.46,1469.823333,4.41
7,Brazil,7,1785.19,1510.733333,3.53
8,Morocco,7,1776.40,1513.663333,3.53
9,Switzerland,7,1676.00,1457.020000,3.40


In [16]:
nombre_archivo = 'forma_actual_mundial_2026.csv'
df_final.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')